# RQ5 — How Does Grad-CAM Attention Evolve During Training?
## XAI Skin Lesion Classification — Research Question 5

**Hypothesis**: Early training focuses on texture/color shortcuts; late training
focuses on clinically meaningful morphological features (lesion border, color asymmetry).

**This is the most novel RQ**. It requires training checkpoints saved at multiple epochs.

**Prerequisites**: You must have checkpoints saved during training.
In your training script, add checkpoint saving every 5 epochs:

```python
if (epoch + 1) % 5 == 0:
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'val_auc': val_auc,
    }, f'outputs/models/checkpoints/{model_name}_epoch{epoch+1}.pth')
```

Expected checkpoint files:
  - `resnet50_epoch5.pth`
  - `resnet50_epoch10.pth`
  - `resnet50_epoch15.pth`
  - `resnet50_epoch20.pth`  (final)

---

## CELL 1: Check Available Checkpoints

In [ ]:
from pathlib import Path
import torch

ckpt_dir = Path('../Skin_Lesion_Classification_backend/ml/outputs/models/checkpoints')
ckpts = sorted(ckpt_dir.glob('resnet50_epoch*.pth'))

print("Available checkpoints:")
for c in ckpts:
    data = torch.load(c, map_location='cpu')
    epoch = data.get('epoch', '?')
    auc = data.get('val_auc', '?')
    print(f"  {c.name}: epoch={epoch}, val_auc={auc:.4f if isinstance(auc, float) else auc}")

if not ckpts:
    print("\u26a0️ No checkpoints found!")
    print("You need to retrain with checkpoint saving enabled.")
    print("Add the checkpoint code in CELL 1 of DL_SCIENTIST_TESTING_GUIDE.md to your train.py")

---

## CELL 2: Compute CAM for Same Image at Each Checkpoint

This is the temporal analysis — watching attention evolve as the model learns.

In [ ]:
import sys
sys.path.append('../Skin_Lesion_Classification_backend/ml')

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from src.models.classifier import create_model, get_target_layer
from src.data.dataset import get_transforms
import matplotlib.cm as cm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = get_transforms('test', 224)

# Load one fixed image — same image throughout
df = pd.read_csv('../Skin_Lesion_Classification_backend/ml/data/processed/metadata_with_paths.csv')
sample = df[df['label'] == 1].sample(1, random_state=99).iloc[0]
orig = np.array(Image.open(sample['filepath']).convert('RGB').resize((224, 224)))
img_float = orig.astype(np.float32) / 255.0
input_t = transform(image=orig)['image'].unsqueeze(0).to(device)

print(f"Fixed image: {sample['image_id']} ({sample['dx']})")

checkpoints = sorted(Path('../Skin_Lesion_Classification_backend/ml/outputs/models/checkpoints').glob('resnet50_epoch*.pth'))

temporal_results = []
cam_maps_temporal = {}

for ckpt_path in checkpoints:
    epoch_num = int(ckpt_path.stem.split('epoch')[1])

    model_t = create_model('resnet50', num_classes=2).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model_t.load_state_dict(ckpt['model_state_dict'])
    model_t.eval()
    target_layer = get_target_layer(model_t, 'resnet50')

    with torch.no_grad():
        out = model_t(input_t)
        probs = torch.softmax(out, dim=1)[0]
        pred = probs.argmax().item()
        conf = probs[pred].item()
        mal_prob = probs[1].item()

    with GradCAM(model=model_t, target_layers=[target_layer]) as cam_gen:
        cam = cam_gen(input_tensor=input_t, targets=[ClassifierOutputTarget(1)])[0]

    cam_maps_temporal[epoch_num] = cam
    temporal_results.append({
        'epoch': epoch_num,
        'val_auc': ckpt.get('val_auc', None),
        'malignant_prob': mal_prob,
        'fap_05': float((cam >= 0.5).sum() / cam.size),
        'entropy': -np.sum((cam.flatten() / (cam.sum() + 1e-8)) * np.log(cam.flatten() / (cam.sum() + 1e-8) + 1e-8)),
        'cam_max_row': np.argmax(cam) // cam.shape[1] / cam.shape[0],
    })

temporal_df = pd.DataFrame(temporal_results)
print("\nTemporal results:")
print(temporal_df)

---

## CELL 3: Visualize Attention Over Time — Your Paper Figure

In [ ]:
epochs = sorted(cam_maps_temporal.keys())
n_epochs = len(epochs)

fig, axes = plt.subplots(3, n_epochs + 1, figsize=(3 * (n_epochs + 1), 9))

# Column 0: original
axes[0][0].imshow(img_float)
axes[0][0].set_title('Original\nImage', fontweight='bold')
axes[0][0].axis('off')
axes[1][0].axis('off')
axes[2][0].axis('off')

for col, epoch_num in enumerate(epochs, start=1):
    cam = cam_maps_temporal[epoch_num]
    meta = temporal_df[temporal_df['epoch'] == epoch_num].iloc[0]

    # Row 0: Heatmap
    axes[0][col].imshow(cm.jet(cam)[:, :, :3])
    auc_str = f"{meta.val_auc:.3f}" if meta.val_auc else 'N/A'
    axes[0][col].set_title(f'Epoch {epoch_num}\nAUC={auc_str}')
    axes[0][col].axis('off')

    # Row 1: Overlay
    heatmap = cm.jet(cam)[:, :, :3]
    overlay = 0.6 * img_float + 0.4 * heatmap
    axes[1][col].imshow(np.clip(overlay, 0, 1))
    axes[1][col].set_title(f'FAP={meta.fap_05:.3f}', fontsize=9)
    axes[1][col].axis('off')

    # Row 2: CAM intensity as histogram
    axes[2][col].hist(cam.flatten(), bins=30, color='steelblue', edgecolor='black', linewidth=0.5)
    axes[2][col].set_title(f'H={meta.entropy:.2f}', fontsize=9)
    axes[2][col].set_xlabel('Activation')

axes[0][0].set_ylabel('Heatmap', fontsize=11)
axes[1][0].set_ylabel('Overlay', fontsize=11)
axes[2][0].set_ylabel('Distribution', fontsize=11)
axes[2][0].axis('off')

plt.suptitle(f'Temporal Grad-CAM — {sample["dx"]} — How attention evolves during training', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/RQ5_temporal_cam.png', dpi=150, bbox_inches='tight')
plt.show()

---

## CELL 4: Metrics Over Epochs — Quantify the Evolution

In [ ]:
import scipy.stats as stats

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(temporal_df['epoch'], temporal_df['val_auc'], 'b-o')
axes[0].set_title('Val AUC over Training')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AUC')
axes[0].grid(alpha=0.3)

axes[1].plot(temporal_df['epoch'], temporal_df['fap_05'], 'r-o')
axes[1].set_title('Focus Area % over Training\n(does attention get more focused?)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Focus Area %')
axes[1].grid(alpha=0.3)

axes[2].plot(temporal_df['epoch'], temporal_df['entropy'], 'g-o')
axes[2].set_title('Entropy over Training\n(does attention concentrate?)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Entropy')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/RQ5_temporal_metrics.png', dpi=150)
plt.show()

# Correlation between accuracy and focus
valid = temporal_df[temporal_df['val_auc'].notna()]
if len(valid) > 2:
    r, p = stats.pearsonr(valid['val_auc'], valid['fap_05'])
    print(f"Correlation (AUC vs FAP): r={r:.4f}, p={p:.4f}")
    if r < 0:
        print("Negative r → higher accuracy = more focused attention → SUPPORTS HYPOTHESIS")
    else:
        print("Positive r → higher accuracy = less focused attention → CONTRADICTS HYPOTHESIS")
else:
    print("Not enough checkpoints with val_auc to compute correlation.")